# Riemannian Embedded Transformer

In [10]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import time

#THE MATH
class PoincareManifold:
    def __init__(self, c=1.0):
        self.c = c  

    def mobius_add(self, x, y):
        x2 = torch.sum(x * x, dim=-1, keepdim=True)
        y2 = torch.sum(y * y, dim=-1, keepdim=True)
        xy = torch.sum(x * y, dim=-1, keepdim=True)
        num = (1 + 2 * self.c * xy + self.c * y2) * x + (1 - self.c * x2) * y
        denom = 1 + 2 * self.c * xy + self.c**2 * x2 * y2
        return num / torch.clamp(denom, min=1e-15)

    def exp_map(self, p, v):
        v_norm = torch.norm(v, p=2, dim=-1, keepdim=True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p * p, dim=-1, keepdim=True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        if isinstance(p, (int, float)) and p == 0:
            p = torch.zeros_like(x)
        x_transformed = self.mobius_add(-p, x)
        sqrt_c = np.sqrt(self.c)
        x_norm = torch.norm(x_transformed, p=2, dim=-1, keepdim=True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p * p, dim=-1, keepdim=True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        arg = (sqrt_c * x_norm).clamp(max=1 - 1e-7)
        scalar = (2 / (lambda_p * sqrt_c * x_norm)) * torch.atanh(arg)
        return scalar * x_transformed

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, p=2, dim=-1)**2
        nx_val = 1 - self.c * torch.norm(x, p=2, dim=-1)**2
        ny_val = 1 - self.c * torch.norm(y, p=2, dim=-1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_val.clamp(min=1e-15) * ny_val.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, p=2, dim=-1, keepdim=True)
        max_norm = (1 - 1e-5) / np.sqrt(self.c)
        cond = norm >= max_norm
        projected = x / norm * max_norm
        return torch.where(cond, projected, x)

#ATTENTION
class MultiHeadPoincareAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, manifold):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.manifold = manifold
        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        b, s, d = x.shape
        q = self.q_linear(x).view(b, s, self.num_heads, self.head_dim)
        k = self.k_linear(x).view(b, s, self.num_heads, self.head_dim)
        v = self.v_linear(x).view(b, s, self.num_heads, self.head_dim)
        q_ball = self.manifold.exp_map(0, q).transpose(1, 2)
        k_ball = self.manifold.exp_map(0, k).transpose(1, 2)
        dists = self.manifold.dist(q_ball.unsqueeze(3), k_ball.unsqueeze(2))
        attn = torch.softmax(-dists / 0.01, dim=-1)
        v_heads = v.transpose(1, 2)
        out_heads = torch.matmul(attn, v_heads) 
        out = out_heads.transpose(1, 2).contiguous().view(b, s, d)
        out_ball = self.manifold.exp_map(0, out)
        return self.out_proj(out_ball)

#TRANSFORMER
class PoincareTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, manifold):
        super().__init__()
        self.attn = MultiHeadPoincareAttention(embed_dim, num_heads, manifold)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim)
        )
        self.ln2 = nn.LayerNorm(embed_dim)
        self.manifold = manifold

    def forward(self, x):
        x_attn = self.attn(self.ln1(x))
        x = self.manifold.exp_map(0, self.manifold.log_map(0, x) + x_attn)
        x_ff = self.ff(self.ln2(x))
        x = self.manifold.exp_map(0, self.manifold.log_map(0, x) + x_ff)
        return x

#OPTIMIZER
class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold):
        defaults = dict(lr=lr, manifold=manifold)
        super().__init__(params, defaults)
    def step(self, closure=None):
        loss = None
        if closure is not None: loss = closure()
        for group in self.param_groups:
            manifold = group['manifold']
            lr = group['lr']
            for p in group['params']:
                if p.grad is None: continue
                p_sq_norm = torch.sum(p.data**2, dim=-1, keepdim=True)
                rescale_factor = ((1 - manifold.c * p_sq_norm)**2) / 4
                riemannian_grad = p.grad.data * rescale_factor
                p.data = manifold.exp_map(p=p.data, v=-lr * riemannian_grad)
                p.data = manifold.project(p.data)
        return loss

#EMBEDDING
G = nx.balanced_tree(r=3, h=4)
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
vocab_size = len(G.nodes())
embed_dim, num_heads = 16, 4
manifold = PoincareManifold(c=1.0)
embedding_layer = nn.Embedding(vocab_size, embed_dim)

with torch.no_grad():
    angles = torch.rand(vocab_size) * 2 * np.pi
    r = 0.3 
    embedding_layer.weight[:, 0] = r * torch.cos(angles)
    embedding_layer.weight[:, 1] = r * torch.sin(angles)
    if embed_dim > 2: embedding_layer.weight[:, 2:] *= 0.1
        
transformer = PoincareTransformerBlock(embed_dim, num_heads, manifold)
optimizer = RSGD(list(embedding_layer.parameters()) + list(transformer.parameters()), lr=5.0, manifold=manifold)

#TRAINING
for epoch in range(1001):
    optimizer.zero_grad()
    all_indices = torch.arange(vocab_size)
    embeds = embedding_layer(all_indices).unsqueeze(0) 
    transformed = transformer(manifold.exp_map(0, embeds))
    u_idx, v_idx = edge_index[:, 0], edge_index[:, 1]
    u_embed, v_embed = transformed[0, u_idx], transformed[0, v_idx]
    dist_pos = manifold.dist(u_embed, v_embed)
    neg_idx = torch.randint(0, vocab_size, (u_idx.size(0),))
    dist_neg = manifold.dist(u_embed, transformed[0, neg_idx])
    root_pos = transformed[0, 0]
    root_penalty = manifold.dist(root_pos, torch.zeros_like(root_pos)).mean()
    loss = dist_pos.mean() - 4.0 * torch.log(dist_neg.mean() + 1e-6) + 0.1 * root_penalty
    loss.backward()
    if epoch < 100: embedding_layer.weight.grad += torch.randn_like(embedding_layer.weight.grad) * 0.05
    torch.nn.utils.clip_grad_norm_(transformer.parameters(), 1.0)
    optimizer.step()
    if epoch == 500: 
        for g in optimizer.param_groups: g['lr'] = 1.0
    if epoch % 200 == 0: print(f"Epoch {epoch}: Loss {loss.item():.4f}")

#TEST
def evaluate_poincare(emb_layer, model, manifold, graph):
    model.eval()
    emb_layer.eval()
    print("\n" + "~~"*30 + "\nRIEMANNIAN PERFORMANCE\n" + "~~"*30)
    
    with torch.no_grad():
        all_indices = torch.arange(vocab_size)
        z = model(manifold.exp_map(0, emb_layer(all_indices).unsqueeze(0))).squeeze(0)
        
        pairs = np.random.choice(vocab_size, (500, 2))
        p_dists, g_dists = [], []
        for u, v in pairs:
            if u == v: continue
            g_dists.append(nx.shortest_path_length(graph, u, v))
            p_dists.append(manifold.dist(z[u], z[v]).item())
        
        corr = np.corrcoef(g_dists, p_dists)[0, 1]
        distortions = [abs(p - g) / max(g, 1e-6) for p, g in zip(p_dists, g_dists)]
        
        start = time.time()
        for _ in range(100): _ = model(manifold.exp_map(0, emb_layer(all_indices).unsqueeze(0)))
        tps = vocab_size / ((time.time() - start) / 100)
        
        print(f"Correlation (r): {corr:.4f}")
        print(f"Avg Distortion:  {np.mean(distortions):.4f}")
        print(f"Max Radius:      {torch.norm(z, p=2, dim=-1).max():.4f}")
        print(f"Throughput:      {tps:.0f} nodes/sec")

evaluate_poincare(embedding_layer, transformer, manifold, G)

Epoch 0: Loss -1.1601
Epoch 200: Loss -4.0374
Epoch 400: Loss -5.0780
Epoch 600: Loss -5.6623
Epoch 800: Loss -5.6609
Epoch 1000: Loss -5.5849

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
RIEMANNIAN PERFORMANCE
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Correlation (r): 0.3672
Avg Distortion:  0.4915
Max Radius:      0.9980
Throughput:      18608 nodes/sec


# Euclidian Embedded Transformer

In [11]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
import time

class StandardTransformerBlock(nn.Module):
    def __init__(self, d_model, nhead):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x

G = nx.balanced_tree(r=3, h=4)
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
vocab_size = len(G.nodes())
d_model, nhead = 16, 4

embedding_layer = nn.Embedding(vocab_size, d_model)
transformer = StandardTransformerBlock(d_model, nhead)

optimizer = torch.optim.Adam(list(embedding_layer.parameters()) + list(transformer.parameters()), lr=0.001)

for epoch in range(1001):
    optimizer.zero_grad()
    
    all_idx = torch.arange(vocab_size)
    x = embedding_layer(all_idx).unsqueeze(0) 
    z = transformer(x).squeeze(0)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    dist_pos = torch.norm(z[u] - z[v], p=2, dim=-1)
    
    neg_idx = torch.randint(0, vocab_size, (u.size(0),))
    dist_neg = torch.norm(z[u] - z[neg_idx], p=2, dim=-1)
    
    loss = dist_pos.mean() - 4.0 * torch.log(dist_neg.mean() + 1e-6) + 0.1 * torch.norm(z[0])
    
    loss.backward()
    optimizer.step()
    
    if epoch % 250 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f}")

def evaluate_euclidean(emb_layer, model, graph):
    model.eval()
    emb_layer.eval()
    print("\n" + "~~"*30 + "\nEUCLIDEAN CONTROL RESULTS\n" + "~~"*30)
    
    with torch.no_grad():
        all_indices = torch.arange(vocab_size)
        z = model(emb_layer(all_indices).unsqueeze(0)).squeeze(0)
        
        pairs = np.random.choice(vocab_size, (500, 2))
        e_dists, g_dists = [], []
        for u, v in pairs:
            if u == v: continue
            g_dists.append(nx.shortest_path_length(graph, u, v))
            e_dists.append(torch.norm(z[u] - z[v], p=2).item())
        
        corr = np.corrcoef(g_dists, e_dists)[0, 1]
        distortions = [abs(e - g) / max(g, 1e-6) for e, g in zip(e_dists, g_dists)]
        
        start = time.time()
        for _ in range(100): _ = model(emb_layer(all_indices).unsqueeze(0))
        tps = vocab_size / ((time.time() - start) / 100)
        
        print(f"Correlation (r): {corr:.4f}")
        print(f"Avg Distortion:  {np.mean(distortions):.4f}")
        print(f"Max Magnitude:   {torch.norm(z, p=2, dim=-1).max():.4f}")
        print(f"Throughput:      {tps:.0f} nodes/sec")

evaluate_euclidean(embedding_layer, transformer, G)

Epoch    0 | Loss: -0.7075
Epoch  250 | Loss: -6.5691
Epoch  500 | Loss: -11.8459
Epoch  750 | Loss: -14.3556
Epoch 1000 | Loss: -13.3428

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
EUCLIDEAN CONTROL RESULTS
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Correlation (r): 0.0396
Avg Distortion:  15.9369
Max Magnitude:   460.3532
Throughput:      80837 nodes/sec
